####Install our package

In [0]:
!pip install /Workspace/Users/gabor.szabo@allianz.com/cubix_data_engineer_capstone-0.2.9-py3-none-any.whl

####Imports

In [0]:

from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest_volume

from cubix_data_engineer_capstone.etl.gold.wide_sales import get_wide_sales
from cubix_data_engineer_capstone.etl.gold.daily_product_category_metrics import get_daily_product_category_metrics
from cubix_data_engineer_capstone.etl.gold.daily_sales_metrics import get_daily_sales_metrics

from cubix_data_engineer_capstone.etl.silver.calendar import get_calendar
from cubix_data_engineer_capstone.etl.silver.customers import get_customers
from cubix_data_engineer_capstone.etl.silver.products import get_products
from cubix_data_engineer_capstone.etl.silver.product_subcategory import get_product_subcategory
from cubix_data_engineer_capstone.etl.silver.product_category import get_product_category
from cubix_data_engineer_capstone.etl.silver.sales import get_sales 
from cubix_data_engineer_capstone.etl.silver.scd import scd1_uc

from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume, write_file_to_volume

####DROP CATALOG to start from scratch

In [0]:
spark.sql("DROP CATALOG capstone CASCADE")

spark.sql("CREATE CATALOG IF NOT EXISTS capstone")
spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS capstone.bronze.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

source_system_tables = [
    "calendar",
    "customers",
    "product_category",
    "product_subcategory",
    "products",
    "sales"
]

for table in source_system_tables:
    dbutils.fs.mkdirs(f"/Volumes/capstone/bronze/bronze/{table}")

####Initial load

In [0]:
datasets = {
    "calendar": {
        "file_name": "calendar.csv",
        "function": get_calendar
    },
    "customers": {
        "file_name": "customers_1.csv",
        "function": get_customers
    },
    "product_category": {
        "file_name": "product_category.csv",
        "function": get_product_category
    },
    "product_subcategory": {
        "file_name": "product_subcategory.csv",
        "function": get_product_subcategory
    },
    "products": {
        "file_name": "products.csv",
        "function": get_products
    },
    "sales": {
        "file_name": "sales_1.csv",
        "function": get_sales
    }
}

# "strategy pattern, algorythm selected on runtime; refactoring.guru/design-patterns/strategy"
for dataset, params in datasets.items():
    print(dataset, params)

    bronze_ingest_volume(
        source_path=f"/Volumes/source_system/source_system/source_files/{dataset}",
        bronze_path=f"/Volumes/capstone/bronze/bronze/{dataset}/",
        file_name=params["file_name"],
    )

    print(f"{dataset} has been processed in the Bronze layer.")

    raw_dataframe = read_file_from_volume(
        full_path=f"/Volumes/capstone/bronze/bronze/{dataset}/{params['file_name']}",
        format="csv"    
    )

    transform_func = params["function"]
    transformed_dataframe = transform_func(raw_dataframe)
    
    (
        transformed_dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"capstone.silver.{dataset}")
    )

    print(f"{dataset} has been processed in the Silver layer")

####Ingestion tasks

In [0]:
batch_2 = {
    "customers": {
        "file_name": "customers_2.csv",
        "primary_key": "CustomerKey",
        "function": get_customers
    },
    "products": {
        "file_name": "products_ingest_1.csv",
        "primary_key": "ProductKey",
        "function": get_products
    },
    "product_subcategory": {
        "file_name": "product_subcategory_ingest_1.csv",
        "primary_key": "ProductSubcategoryKey",
        "function": get_product_subcategory
    },
    "product_category": {
        "file_name": "product_category_ingest_1.csv",
        "primary_key": "ProductCategoryKey",
        "function": get_product_category
    },
    "sales": {
        "file_name": "sales_202405.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

batch_3 = {
    "customers": {
        "file_name": "customers_3.csv",
        "primary_key": "CustomerKey",
        "function": get_customers
    },
    "sales": {
        "file_name": "sales_202406.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

batch_4 = {
    "customers": {
        "file_name": "customers_4.csv",
        "primary_key": "CustomerKey",
        "function": get_customers

    },
    "sales": {
        "file_name": "sales_202407.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

ingestion_job = batch_4

####Bronze ingestion

In [0]:
for dataset_key, params in ingestion_job.items():

    bronze_ingest_volume(
        source_path=f"/Volumes/source_system/source_system/source_files/{dataset_key}",
        bronze_path=f"/Volumes/capstone/bronze/bronze/{dataset_key}",
        file_name=params["file_name"],
    )

    print(f"{dataset_key} ({params['file_name']}) ingested")

####Silver ingestion

In [0]:
for dataset, params in ingestion_job.items():

    raw_dataframe = read_file_from_volume(
        full_path=f"/Volumes/capstone/bronze/bronze/{dataset}/{params['file_name']}",
        format="csv"
    )

    transform_func = params["function"]
    transformed_dataframe = transform_func(raw_dataframe)

    # sales is not SCD, it's a Fact table, therefore appending.
    if dataset == "sales":
        (
            transformed_dataframe
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(f"capstone.silver.{dataset}")
        )
    else:
        scd1_uc(spark, f"capstone.silver.{dataset}", transformed_dataframe, primary_key=params["primary_key"])

    print(f"{dataset} has been processed in the Silver layer.")

####Gold layer

In [0]:
calendar_master = spark.table("capstone.silver.calendar")
customers_master = spark.table("capstone.silver.customers")
product_subcategory_master = spark.table("capstone.silver.product_subcategory")
product_category_master = spark.table("capstone.silver.product_category")
products_master = spark.table("capstone.silver.products")
sales_master = spark.table("capstone.silver.sales")

####Wide Sales

In [0]:
wide_sales_df = get_wide_sales(
    sales_master=sales_master,
    calendar_master=calendar_master,
    customers_master=customers_master,
    products_master=products_master,
    product_subcategory_master=product_subcategory_master,
    product_category_master=product_category_master
)

In [0]:
wide_sales_df.count()

In [0]:
(
    wide_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.wide_sales")
)

####Daily Product Category Metrics

In [0]:
daily_product_category_metrics = get_daily_product_category_metrics(wide_sales_df)

In [0]:
(
    daily_product_category_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_product_category_metrics")
)

####Daily Sales Metrics

In [0]:
daily_sales_metrics = get_daily_sales_metrics(wide_sales_df) 

In [0]:
(
    daily_sales_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_sales_metrics")
)

####Write parquet file after 4th batch

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS capstone.gold.gold")

In [0]:
wide_sales_df = spark.table("capstone.gold.wide_sales")

In [0]:
write_file_to_volume(
    df=wide_sales_df,
    full_path="/Volumes/capstone/gold/gold/wide_sales.parquet",
    format="parquet",
    mode="overwrite"
)